In [ ]:
# import numpy as np
# import pandas as pd
# from datetime import datetime, timedelta
# from typing import Dict, List, Optional, Tuple
# from dataclasses import dataclass, field
# from enum import Enum
# import warnings
# from abc import ABC, abstractmethod
# from datetime import datetime
# import pytz
# warnings.filterwarnings('ignore')


# # =============================================================================
# # STEP 1: CONFIGURATION CLASSES
# # =============================================================================

# @dataclass
# class MarginConfig:
#     """Configuration for margin calculation"""
#     var_margin_percent: float = 12.0
#     elm_percent: float = 3.0
#     span_multiplier: float = 1.0
#     min_margin_per_lot: float = 50000

#     def calculate_span_margin(self, spot_price: float, lot_size: int,
#                               option_price: float, quantity: int,
#                               is_short: bool = True) -> float:
#         if is_short:
#             base_margin = spot_price * lot_size * (self.var_margin_percent / 100)
#             elm_margin = spot_price * lot_size * (self.elm_percent / 100)
#             total_base = (base_margin + elm_margin) * abs(quantity) / lot_size
#             total_base *= self.span_multiplier
#             premium_received = option_price * abs(quantity)
#             margin = max(total_base - premium_received,
#                          self.min_margin_per_lot * abs(quantity) / lot_size)
#         else:
#             margin = option_price * abs(quantity)
#         return max(margin, 0)


# @dataclass
# class StrategyParams:
#     """OSTRAD Strategy Parameters"""
#     time_frame: int = 5
#     start_time: str = "09:25:00"
#     entry_end_time: str = "14:30:00"
#     sl_modification_time: str = "10:00:00"
#     square_off_time: str = "15:20:00"
#     threshold_percentage: float = 10.0
#     sl_percentage: float = 30.0
#     sl_diff_percentage: float = 2.0
#     max_open_strikes_per_leg: int = 3
#     num_strikes: int = 3
#     capital_per_symbol: float = 10_000_000
#     new_position_margin_limit: float = 75.0
#     hedge_margin_limit: float = 60.0
#     hedge_strike_diff_percent: float = 2.5
#     hedge_strike_diff_percent_2: float = 5.0
#     min_premium_percent: float = 0.055
#     symbols: List[str] = field(default_factory=lambda: ['NIFTY'])
#     active_days_to_expiry: List[int] = field(default_factory=lambda: [0, 1, 2])
#     lot_sizes: Dict[str, int] = field(default_factory=lambda: {'NIFTY': 75})
#     strike_diffs: Dict[str, float] = field(default_factory=lambda: {'NIFTY': 50.0})
#     margin_config: MarginConfig = field(default_factory=MarginConfig)


# class DataLoader(ABC):
#     @abstractmethod
#     def load_data(self, symbol: str, start_date: datetime,
#                   end_date: datetime) -> Dict[str, pd.DataFrame]:
#         pass


# class DataFrameLoader(DataLoader):
#     """Load data from provided DataFrames"""

#     def __init__(self, spot_df: pd.DataFrame, options_df: pd.DataFrame):
#         self.spot_df = spot_df.copy()
#         self.options_df = options_df.copy()

#         # Ensure datetime index
#         if not isinstance(self.spot_df.index, pd.DatetimeIndex):
#             if 'date' in self.spot_df.columns:
#                 self.spot_df.set_index('date', inplace=True)

#         if not isinstance(self.options_df.index, pd.DatetimeIndex):
#             if 'date' in self.options_df.columns:
#                 self.options_df.set_index('date', inplace=True)

#         # FIX: Strip timezone info from both DataFrames to ensure tz-naive throughout
#         if self.spot_df.index.tz is not None:
#             self.spot_df.index = self.spot_df.index.tz_localize(None)

#         if self.options_df.index.tz is not None:
#             self.options_df.index = self.options_df.index.tz_localize(None)

#         # FIX: Strip tz from expiry column if present
#         if 'expiry' in self.options_df.columns:
#             self.options_df['expiry'] = pd.to_datetime(
#                 self.options_df['expiry']).dt.tz_localize(None)

#     def load_data(self, symbol: str, start_date: datetime,
#                   end_date: datetime) -> Dict[str, pd.DataFrame]:

#         # FIX: Normalize start/end dates to tz-naive for safe comparison
#         start_ts = pd.Timestamp(start_date).tz_localize(None) \
#             if pd.Timestamp(start_date).tz is None \
#             else pd.Timestamp(start_date).tz_convert(None)
#         end_ts = pd.Timestamp(end_date).tz_localize(None) \
#             if pd.Timestamp(end_date).tz is None \
#             else pd.Timestamp(end_date).tz_convert(None)

#         # Extend end_ts to end of day so all candles for the date are included
#         end_ts_day = end_ts.normalize() + pd.Timedelta(hours=23, minutes=59, seconds=59)

#         spot = self.spot_df[
#             (self.spot_df.index >= start_ts) &
#             (self.spot_df.index <= end_ts_day)
#         ].copy()

#         options = self.options_df[
#             (self.options_df.index >= start_ts) &
#             (self.options_df.index <= end_ts_day)
#         ].copy()

#         if 'symbol' in options.columns:
#             options = options[options['symbol'] == symbol]

#         return {'spot': spot, 'options': options}


# # =============================================================================
# # STEP 2: VWAP INDICATOR
# # =============================================================================

# class VWAPIndicator:
#     @staticmethod
#     def calculate(df: pd.DataFrame, buy_threshold: float = 10.0,
#                   sell_threshold: float = 10.0) -> pd.DataFrame:
#         _temp = df.copy()
#         _temp['traded_volume'] = _temp['close'] * _temp['volume']
#         _temp['indicator'] = _temp['traded_volume'].cumsum() / _temp['volume'].cumsum()
#         _temp['diff'] = _temp['close'] - _temp['indicator']
#         _temp['order_side'] = _temp.apply(
#             lambda x: 'buy' if x['diff'] > 0 else 'sell', axis=1)
#         _temp['perc_diff'] = round(abs(_temp['diff'] / _temp['indicator']) * 100, 2)
#         _temp['can_trade'] = _temp.apply(
#             lambda x: True if (
#                 (x['order_side'] == 'buy' and x['perc_diff'] <= buy_threshold) or
#                 (x['order_side'] == 'sell' and x['perc_diff'] <= sell_threshold)
#             ) else False, axis=1)
#         _temp['signal'] = _temp.apply(
#             lambda x: x['order_side'] if x['can_trade'] else np.nan, axis=1)
#         return _temp


# # =============================================================================
# # STEP 3: POSITION CLASS
# # =============================================================================

# @dataclass
# class Position:
#     symbol: str
#     strike: float
#     option_type: str
#     expiry: datetime
#     entry_time: datetime
#     entry_price: float
#     quantity: int
#     lot_size: int
#     sl_price: Optional[float] = None
#     sl2_price: Optional[float] = None
#     sl_order_placed: bool = False
#     sl_hit: bool = False
#     is_hedge: bool = False
#     hedge_quantity: int = 0
#     exit_time: Optional[datetime] = None
#     exit_price: Optional[float] = None
#     exit_reason: Optional[str] = None
#     pnl: float = 0.0

#     @property
#     def is_short(self) -> bool:
#         return self.quantity < 0

#     @property
#     def is_open(self) -> bool:
#         return self.exit_time is None

#     @property
#     def lots(self) -> int:
#         return abs(self.quantity) // self.lot_size


# # =============================================================================
# # STEP 4: BACKTESTER CLASS
# # =============================================================================

# class OSTRADBacktester:
#     def __init__(self, params: StrategyParams, data_loader: DataLoader):
#         self.params = params
#         self.data_loader = data_loader
#         self.margin_config = params.margin_config
#         self.positions: Dict[str, List[Position]] = {s: [] for s in params.symbols}
#         self.closed_positions: List[Position] = []
#         self.traded_strikes: Dict[str, Dict[str, bool]] = {s: {} for s in params.symbols}
#         self.spot_data: Dict[str, pd.DataFrame] = {}
#         self.options_data: Dict[str, pd.DataFrame] = {}
#         self.straddle_data: Dict[str, Dict[str, pd.DataFrame]] = {}
#         self.daily_pnl: Dict[datetime, float] = {}
#         self.margin_history: List[Dict] = []
#         self.trade_log: List[Dict] = []

#     def load_market_data(self, symbol: str, start_date: datetime, end_date: datetime) -> None:
#         data = self.data_loader.load_data(symbol, start_date, end_date)
#         self.spot_data[symbol] = data['spot']
#         self.options_data[symbol] = data['options']
#         self._prepare_straddle_data(symbol)
#         print(f"  Loaded {len(self.spot_data[symbol])} spot candles")
#         print(f"  Loaded {len(self.options_data[symbol])} option records")
#         print(f"  Prepared {len(self.straddle_data.get(symbol, {}))} straddle combinations")

#     def _prepare_straddle_data(self, symbol: str) -> None:
#         options_df = self.options_data[symbol]
#         if options_df.empty:
#             self.straddle_data[symbol] = {}
#             return

#         expiries = options_df['expiry'].unique()
#         straddle_data = {}

#         for expiry in expiries:
#             expiry_options = options_df[options_df['expiry'] == expiry]
#             strikes = expiry_options['strike'].unique()

#             for strike in strikes:
#                 ce_data = expiry_options[
#                     (expiry_options['strike'] == strike) &
#                     (expiry_options['option_type'] == 'CE')
#                 ].copy()
#                 pe_data = expiry_options[
#                     (expiry_options['strike'] == strike) &
#                     (expiry_options['option_type'] == 'PE')
#                 ].copy()

#                 if ce_data.empty or pe_data.empty:
#                     continue

#                 merged = pd.merge(
#                     ce_data[['close', 'volume', 'high', 'low']].rename(
#                         columns={'close': 'ce_close', 'volume': 'ce_volume',
#                                  'high': 'ce_high', 'low': 'ce_low'}),
#                     pe_data[['close', 'volume', 'high', 'low']].rename(
#                         columns={'close': 'pe_close', 'volume': 'pe_volume',
#                                  'high': 'pe_high', 'low': 'pe_low'}),
#                     left_index=True, right_index=True, how='inner')

#                 if merged.empty:
#                     continue

#                 merged['close'] = merged['ce_close'] + merged['pe_close']
#                 merged['high'] = merged['ce_high'] + merged['pe_high']
#                 merged['low'] = merged['ce_low'] + merged['pe_low']
#                 merged['volume'] = merged[['ce_volume', 'pe_volume']].min(axis=1)

#                 straddle_vwap = VWAPIndicator.calculate(
#                     merged[['close', 'high', 'low', 'volume']].reset_index(),
#                     buy_threshold=self.params.threshold_percentage,
#                     sell_threshold=self.params.threshold_percentage)

#                 if 'date' in straddle_vwap.columns:
#                     straddle_vwap.set_index('date', inplace=True)

#                 straddle_vwap['ce_close'] = merged['ce_close']
#                 straddle_vwap['pe_close'] = merged['pe_close']
#                 straddle_vwap['ce_high'] = merged['ce_high']
#                 straddle_vwap['pe_high'] = merged['pe_high']

#                 key = f"{expiry}_{strike}"
#                 straddle_data[key] = straddle_vwap

#         self.straddle_data[symbol] = straddle_data

#     def get_atm_strike(self, symbol: str, timestamp: datetime) -> float:
#         spot_df = self.spot_data[symbol]
#         # FIX: Use closest available candle at or before timestamp
#         spot_data = spot_df[spot_df.index <= timestamp]
#         if spot_data.empty:
#             return 0
#         spot_price = spot_data['close'].iloc[-1]
#         strike_diff = self.params.strike_diffs.get(symbol, 50)
#         return round(spot_price / strike_diff) * strike_diff

#     def get_spot_price(self, symbol: str, timestamp: datetime) -> float:
#         spot_df = self.spot_data[symbol]
#         # FIX: Use closest available candle at or before timestamp
#         spot_data = spot_df[spot_df.index <= timestamp]
#         if spot_data.empty:
#             return 0
#         return spot_data['close'].iloc[-1]

#     def get_option_price(self, symbol: str, strike: float, option_type: str,
#                          expiry: datetime, timestamp: datetime,
#                          price_type: str = 'close') -> float:
#         options_df = self.options_data[symbol]
#         price_data = options_df[
#             (options_df['strike'] == strike) &
#             (options_df['option_type'] == option_type) &
#             (options_df['expiry'] == expiry) &
#             (options_df.index <= timestamp)]
#         if price_data.empty:
#             return 0.0
#         return price_data[price_type].iloc[-1]

#     def get_nearest_expiry(self, symbol: str, timestamp: datetime) -> Optional[datetime]:
#         options_df = self.options_data[symbol]
#         # FIX: Filter by expiry date (not index timestamp) >= today's date
#         today = pd.Timestamp(timestamp).normalize()
#         expiries = options_df[options_df['expiry'] >= today]['expiry'].unique()
#         if len(expiries) == 0:
#             return None
#         return min(expiries)

#     def calculate_margin(self, symbol: str, timestamp: datetime) -> Dict:
#         total_margin = 0.0
#         breakdown = {}
#         spot_price = self.get_spot_price(symbol, timestamp)
#         lot_size = self.params.lot_sizes.get(symbol, 75)

#         for pos in self.positions[symbol]:
#             if not pos.is_open:
#                 continue
#             opt_price = self.get_option_price(
#                 symbol, pos.strike, pos.option_type, pos.expiry, timestamp)
#             margin = self.margin_config.calculate_span_margin(
#                 spot_price=spot_price, lot_size=lot_size, option_price=opt_price,
#                 quantity=pos.quantity, is_short=pos.is_short)
#             key = f"{pos.strike}_{pos.option_type}"
#             breakdown[key] = margin
#             total_margin += margin

#         margin_percent = (total_margin / self.params.capital_per_symbol) * 100
#         return {
#             'total_margin': total_margin,
#             'margin_percent': margin_percent,
#             'breakdown': breakdown
#         }

#     def check_signals(self, symbol: str, timestamp: datetime) -> List[Dict]:
#         signals = []
#         atm = self.get_atm_strike(symbol, timestamp)
#         if atm == 0:
#             return signals

#         strike_diff = self.params.strike_diffs.get(symbol, 50)
#         expiry = self.get_nearest_expiry(symbol, timestamp)
#         if expiry is None:
#             return signals

#         for strike_offset in range(-self.params.num_strikes, self.params.num_strikes + 1):
#             strike = atm + (strike_offset * strike_diff)
#             key = f"{expiry}_{strike}"
#             strike_key = f"{strike}_{expiry.date() if hasattr(expiry, 'date') else expiry}"
#             if self.traded_strikes[symbol].get(strike_key):
#                 continue

#             if key not in self.straddle_data[symbol]:
#                 continue

#             straddle_df = self.straddle_data[symbol][key]
#             current_data = straddle_df[straddle_df.index <= timestamp]

#             if len(current_data) < 2:
#                 continue

#             prev_candle = current_data.iloc[-2]

#             if prev_candle['signal'] == 'sell' and prev_candle['can_trade']:
#                 spot_price = self.get_spot_price(symbol, timestamp)
#                 min_premium = spot_price * (self.params.min_premium_percent / 100)
#                 ce_price = prev_candle.get('ce_close', 0)
#                 pe_price = prev_candle.get('pe_close', 0)

#                 if ce_price > min_premium:
#                     signals.append({
#                         'strike': strike, 'option_type': 'CE', 'expiry': expiry,
#                         'signal_type': 'SELL', 'straddle_price': prev_candle['close'],
#                         'vwap': prev_candle['indicator'], 'deviation': prev_candle['perc_diff'],
#                         'entry_price': ce_price, 'timestamp': timestamp})

#                 if pe_price > min_premium:
#                     signals.append({
#                         'strike': strike, 'option_type': 'PE', 'expiry': expiry,
#                         'signal_type': 'SELL', 'straddle_price': prev_candle['close'],
#                         'vwap': prev_candle['indicator'], 'deviation': prev_candle['perc_diff'],
#                         'entry_price': pe_price, 'timestamp': timestamp})

#         return signals

#     def take_position(self, symbol: str, signal: Dict, timestamp: datetime) -> Optional[Position]:
#         margin_info = self.calculate_margin(symbol, timestamp)
#         if margin_info['margin_percent'] >= self.params.new_position_margin_limit:
#             self._log_trade(timestamp, symbol, signal['strike'], signal['option_type'],
#                             'ENTRY_REJECTED', 'Margin limit exceeded', 0, 0)
#             return None

#         open_positions = [p for p in self.positions[symbol] if p.is_open and not p.is_hedge]
#         ce_count = len([p for p in open_positions if p.option_type == 'CE'])
#         pe_count = len([p for p in open_positions if p.option_type == 'PE'])

#         if signal['option_type'] == 'CE' and ce_count >= self.params.max_open_strikes_per_leg:
#             return None
#         if signal['option_type'] == 'PE' and pe_count >= self.params.max_open_strikes_per_leg:
#             return None

#         lot_size = self.params.lot_sizes.get(symbol, 75)
#         entry_price = self.get_option_price(
#             symbol, signal['strike'], signal['option_type'],
#             signal['expiry'], timestamp)
#         if entry_price <= 0:
#             return None

#         margin_per_lot = self.margin_config.calculate_span_margin(
#             spot_price=self.get_spot_price(symbol, timestamp), lot_size=lot_size,
#             option_price=entry_price, quantity=-lot_size, is_short=True)

#         if margin_per_lot > 0:
#             max_lots = int(
#                 (self.params.capital_per_symbol * 0.5) / margin_per_lot /
#                 (self.params.num_strikes * 2 + 1))
#             lots = max(1, min(max_lots, 2))
#         else:
#             lots = 1

#         quantity = -lots * lot_size
#         sl_price = entry_price * (1 + self.params.sl_percentage / 100)

#         position = Position(
#             symbol=symbol, strike=signal['strike'], option_type=signal['option_type'],
#             expiry=signal['expiry'], entry_time=timestamp, entry_price=entry_price,
#             quantity=quantity, lot_size=lot_size, sl_price=sl_price, sl_order_placed=True)

#         self.positions[symbol].append(position)
#         expiry_date = signal['expiry'].date() if hasattr(signal['expiry'], 'date') else signal['expiry']
#         strike_key = f"{signal['strike']}_{expiry_date}"
#         self.traded_strikes[symbol][strike_key] = True

#         self._log_trade(timestamp, symbol, signal['strike'], signal['option_type'],
#                         'ENTRY', f"SL at {sl_price:.2f}", entry_price, quantity)
#         return position

#     def check_and_take_hedge(self, symbol: str, timestamp: datetime) -> List[Position]:
#         hedges = []
#         margin_info = self.calculate_margin(symbol, timestamp)
#         if margin_info['margin_percent'] < self.params.hedge_margin_limit:
#             return hedges

#         open_shorts = [p for p in self.positions[symbol]
#                        if p.is_open and p.is_short and not p.is_hedge]
#         if not open_shorts:
#             return hedges

#         spot_price = self.get_spot_price(symbol, timestamp)
#         strike_diff = self.params.strike_diffs.get(symbol, 50)
#         lot_size = self.params.lot_sizes.get(symbol, 75)
#         expiry = self.get_nearest_expiry(symbol, timestamp)
#         if expiry is None:
#             return hedges

#         ce_short_qty = sum(abs(p.quantity) for p in open_shorts if p.option_type == 'CE')
#         pe_short_qty = sum(abs(p.quantity) for p in open_shorts if p.option_type == 'PE')

#         existing_hedges = [p for p in self.positions[symbol] if p.is_open and p.is_hedge]
#         ce_hedge_qty = sum(p.quantity for p in existing_hedges if p.option_type == 'CE')
#         pe_hedge_qty = sum(p.quantity for p in existing_hedges if p.option_type == 'PE')

#         is_first_hedge = len(existing_hedges) == 0
#         hedge_pct = (self.params.hedge_strike_diff_percent if is_first_hedge
#                      else self.params.hedge_strike_diff_percent_2)

#         ce_hedge_strike = round(spot_price * (1 + hedge_pct / 100) / strike_diff) * strike_diff
#         pe_hedge_strike = round(spot_price * (1 - hedge_pct / 100) / strike_diff) * strike_diff

#         if ce_short_qty > 0 and ce_hedge_qty < ce_short_qty // 2:
#             hedge_qty = ce_short_qty // 2 - ce_hedge_qty
#             hedge_qty = max(lot_size, (hedge_qty // lot_size) * lot_size)
#             entry_price = self.get_option_price(symbol, ce_hedge_strike, 'CE', expiry, timestamp)
#             if entry_price > 0:
#                 hedge_pos = Position(
#                     symbol=symbol, strike=ce_hedge_strike, option_type='CE',
#                     expiry=expiry, entry_time=timestamp, entry_price=entry_price,
#                     quantity=int(hedge_qty), lot_size=lot_size,
#                     is_hedge=True, hedge_quantity=int(hedge_qty))
#                 self.positions[symbol].append(hedge_pos)
#                 hedges.append(hedge_pos)
#                 self._log_trade(timestamp, symbol, ce_hedge_strike, 'CE',
#                                 'HEDGE_ENTRY', 'BUY hedge', entry_price, hedge_qty)

#         if pe_short_qty > 0 and pe_hedge_qty < pe_short_qty // 2:
#             hedge_qty = pe_short_qty // 2 - pe_hedge_qty
#             hedge_qty = max(lot_size, (hedge_qty // lot_size) * lot_size)
#             entry_price = self.get_option_price(symbol, pe_hedge_strike, 'PE', expiry, timestamp)
#             if entry_price > 0:
#                 hedge_pos = Position(
#                     symbol=symbol, strike=pe_hedge_strike, option_type='PE',
#                     expiry=expiry, entry_time=timestamp, entry_price=entry_price,
#                     quantity=int(hedge_qty), lot_size=lot_size,
#                     is_hedge=True, hedge_quantity=int(hedge_qty))
#                 self.positions[symbol].append(hedge_pos)
#                 hedges.append(hedge_pos)
#                 self._log_trade(timestamp, symbol, pe_hedge_strike, 'PE',
#                                 'HEDGE_ENTRY', 'BUY hedge', entry_price, hedge_qty)

#         return hedges

#     def check_sl_hits(self, symbol: str, timestamp: datetime) -> List[Position]:
#         hit_positions = []
#         for pos in self.positions[symbol]:
#             if not pos.is_open or not pos.is_short or pos.sl_hit:
#                 continue
#             if pos.sl_price is None:
#                 continue
#             current_price = self.get_option_price(
#                 symbol, pos.strike, pos.option_type, pos.expiry, timestamp)
#             if current_price <= 0:
#                 continue
#             if current_price >= pos.sl_price:
#                 pos.sl_hit = True
#                 pos.exit_time = timestamp
#                 pos.exit_price = current_price
#                 pos.exit_reason = 'SL_HIT'
#                 pos.pnl = (pos.entry_price - pos.exit_price) * abs(pos.quantity)
#                 hit_positions.append(pos)
#                 self.closed_positions.append(pos)
#                 self._log_trade(timestamp, symbol, pos.strike, pos.option_type,
#                                 'SL_HIT', f'Exited at {current_price:.2f}',
#                                 current_price, pos.quantity)
#         return hit_positions

#     def modify_sl_to_high(self, symbol: str, timestamp: datetime) -> List[Position]:
#         modified = []
#         for pos in self.positions[symbol]:
#             if not pos.is_open or not pos.is_short or pos.sl_hit:
#                 continue
#             if pos.sl2_price is not None:
#                 continue
#             options_df = self.options_data[symbol]
#             day_data = options_df[
#                 (options_df['strike'] == pos.strike) &
#                 (options_df['option_type'] == pos.option_type) &
#                 (options_df['expiry'] == pos.expiry) &
#                 (options_df.index.date == timestamp.date()) &
#                 (options_df.index <= timestamp)]
#             if day_data.empty:
#                 continue
#             day_high = day_data['high'].max()
#             if pos.sl_price and day_high < pos.sl_price:
#                 pos.sl2_price = day_high + 1
#                 pos.sl_price = pos.sl2_price
#                 modified.append(pos)
#                 self._log_trade(timestamp, symbol, pos.strike, pos.option_type,
#                                 'SL_MODIFIED', f'New SL: {pos.sl_price:.2f}', pos.sl_price, 0)
#         return modified

#     def run_rms_checks(self, symbol: str, timestamp: datetime) -> List[Position]:
#         rms_exits = []
#         atm = self.get_atm_strike(symbol, timestamp)
#         strike_diff = self.params.strike_diffs.get(symbol, 50)
#         min_valid_strike = atm - (self.params.num_strikes * strike_diff)
#         max_valid_strike = atm + (self.params.num_strikes * strike_diff)

#         for pos in self.positions[symbol]:
#             if not pos.is_open or pos.is_hedge:
#                 continue
#             should_exit = False
#             exit_reason = None

#             if pos.quantity > 0:
#                 should_exit = True
#                 exit_reason = 'RMS_POSITIVE_QTY'
#             elif pos.strike < min_valid_strike or pos.strike > max_valid_strike:
#                 should_exit = True
#                 exit_reason = 'RMS_INVALID_STRIKE'

#             if should_exit:
#                 current_price = self.get_option_price(
#                     symbol, pos.strike, pos.option_type, pos.expiry, timestamp)
#                 pos.exit_time = timestamp
#                 pos.exit_price = current_price
#                 pos.exit_reason = exit_reason
#                 if pos.is_short:
#                     pos.pnl = (pos.entry_price - pos.exit_price) * abs(pos.quantity)
#                 else:
#                     pos.pnl = (pos.exit_price - pos.entry_price) * pos.quantity
#                 rms_exits.append(pos)
#                 self.closed_positions.append(pos)
#                 self._log_trade(timestamp, symbol, pos.strike, pos.option_type,
#                                 exit_reason, 'RMS squareoff', current_price, pos.quantity)
#         return rms_exits

#     def square_off_all(self, symbol: str, timestamp: datetime) -> List[Position]:
#         squared_off = []
#         shorts = [p for p in self.positions[symbol] if p.is_open and p.is_short]
#         for pos in shorts:
#             exit_price = self.get_option_price(
#                 symbol, pos.strike, pos.option_type, pos.expiry, timestamp)
#             pos.exit_time = timestamp
#             pos.exit_price = exit_price
#             pos.exit_reason = 'EOD_SQUAREOFF'
#             pos.pnl = (pos.entry_price - exit_price) * abs(pos.quantity)
#             squared_off.append(pos)
#             self.closed_positions.append(pos)
#             self._log_trade(timestamp, symbol, pos.strike, pos.option_type,
#                             'EOD_SQUAREOFF', 'Short closed', exit_price, pos.quantity)

#         longs = [p for p in self.positions[symbol] if p.is_open and not p.is_short]
#         for pos in longs:
#             exit_price = self.get_option_price(
#                 symbol, pos.strike, pos.option_type, pos.expiry, timestamp)
#             pos.exit_time = timestamp
#             pos.exit_price = exit_price
#             pos.exit_reason = 'EOD_SQUAREOFF'
#             pos.pnl = (pos.exit_price - pos.entry_price) * pos.quantity
#             squared_off.append(pos)
#             self.closed_positions.append(pos)
#             self._log_trade(timestamp, symbol, pos.strike, pos.option_type,
#                             'EOD_SQUAREOFF', 'Hedge closed', exit_price, pos.quantity)
#         return squared_off

#     def run_backtest(self, start_date: datetime, end_date: datetime) -> Dict:
#         print("=" * 60)
#         print("OSTRAD BACKTEST")
#         print("=" * 60)

#         results = {
#             'trades': [], 'daily_pnl': {},
#             'margin_history': [], 'statistics': {}
#         }

#         for symbol in self.params.symbols:
#             print(f"\nProcessing {symbol}...")
#             self.load_market_data(symbol, start_date, end_date)
#             print(f"Running backtest for {symbol}...")
#             symbol_results = self._run_symbol_backtest(symbol, start_date, end_date)
#             results['trades'].extend(symbol_results['trades'])
#             for date, pnl in symbol_results['daily_pnl'].items():
#                 if date not in results['daily_pnl']:
#                     results['daily_pnl'][date] = 0
#                 results['daily_pnl'][date] += pnl
#             results['margin_history'].extend(symbol_results['margin_history'])

#         results['statistics'] = self._calculate_statistics(results)
#         return results

#     def _run_symbol_backtest(self, symbol: str, start_date: datetime,
#                              end_date: datetime) -> Dict:
#         trades = []
#         daily_pnl = {}
#         margin_history = []

#         # FIX: Normalize to tz-naive date for iteration
#         start_ts = pd.Timestamp(start_date).tz_localize(None) \
#             if pd.Timestamp(start_date).tz is None \
#             else pd.Timestamp(start_date).tz_convert(None)
#         end_ts = pd.Timestamp(end_date).tz_localize(None) \
#             if pd.Timestamp(end_date).tz is None \
#             else pd.Timestamp(end_date).tz_convert(None)

#         current_date = start_ts.date()
#         end_date_only = end_ts.date()

#         while current_date <= end_date_only:
#             if current_date.weekday() >= 5:
#                 current_date += timedelta(days=1)
#                 continue

#             # Reset per-day state
#             self.traded_strikes[symbol] = {}
#             self.positions[symbol] = []   # FIX: reset open positions each day
#             day_pnl = 0.0

#             day_start = datetime.combine(
#                 current_date,
#                 datetime.strptime(self.params.start_time, "%H:%M:%S").time())
#             entry_end = datetime.combine(
#                 current_date,
#                 datetime.strptime(self.params.entry_end_time, "%H:%M:%S").time())
#             sl_mod_time = datetime.combine(
#                 current_date,
#                 datetime.strptime(self.params.sl_modification_time, "%H:%M:%S").time())
#             eod_time = datetime.combine(
#                 current_date,
#                 datetime.strptime(self.params.square_off_time, "%H:%M:%S").time())

#             # FIX: timestamps are tz-naive, matching stripped index
#             timestamps = pd.date_range(
#                 start=day_start, end=eod_time,
#                 freq=f'{self.params.time_frame}min')

#             for timestamp in timestamps:
#                 # FIX: Check if any spot data exists at or before this timestamp
#                 if symbol not in self.spot_data:
#                     continue
#                 available_spot = self.spot_data[symbol][
#                     self.spot_data[symbol].index <= timestamp]
#                 if available_spot.empty:
#                     continue

#                 rms_exits = self.run_rms_checks(symbol, timestamp)
#                 for pos in rms_exits:
#                     day_pnl += pos.pnl
#                     trades.append(self._position_to_dict(pos))

#                 hedges = self.check_and_take_hedge(symbol, timestamp)
#                 for hedge in hedges:
#                     trades.append(self._position_to_dict(hedge))

#                 if timestamp <= entry_end:
#                     signals = self.check_signals(symbol, timestamp)
#                     for signal in signals:
#                         pos = self.take_position(symbol, signal, timestamp)
#                         if pos:
#                             trades.append(self._position_to_dict(pos))

#                 if timestamp >= sl_mod_time:
#                     modified = self.modify_sl_to_high(symbol, timestamp)
#                     for mod in modified:
#                         trades.append(self._position_to_dict(mod))

#                 hits = self.check_sl_hits(symbol, timestamp)
#                 for hit in hits:
#                     day_pnl += hit.pnl
#                     trades.append(self._position_to_dict(hit))

#                 margin_info = self.calculate_margin(symbol, timestamp)
#                 margin_history.append({
#                     'timestamp': timestamp, 'symbol': symbol,
#                     'total_margin': margin_info['total_margin'],
#                     'margin_percent': margin_info['margin_percent']})

#             squared = self.square_off_all(symbol, eod_time)
#             for sq in squared:
#                 day_pnl += sq.pnl
#                 trades.append(self._position_to_dict(sq))

#             daily_pnl[current_date] = day_pnl
#             print(f"  {current_date}: P&L = ₹{day_pnl:,.2f}")
#             current_date += timedelta(days=1)

#         return {'trades': trades, 'daily_pnl': daily_pnl, 'margin_history': margin_history}

#     def _position_to_dict(self, pos: Position) -> Dict:
#         return {
#             'symbol': pos.symbol, 'strike': pos.strike, 'option_type': pos.option_type,
#             'expiry': pos.expiry, 'entry_time': pos.entry_time, 'entry_price': pos.entry_price,
#             'quantity': pos.quantity, 'lots': pos.lots, 'sl_price': pos.sl_price,
#             'sl2_price': pos.sl2_price, 'is_hedge': pos.is_hedge, 'exit_time': pos.exit_time,
#             'exit_price': pos.exit_price, 'exit_reason': pos.exit_reason, 'pnl': pos.pnl}

#     def _log_trade(self, timestamp: datetime, symbol: str, strike: float,
#                    option_type: str, event: str, message: str,
#                    price: float, quantity: int) -> None:
#         self.trade_log.append({
#             'timestamp': timestamp, 'symbol': symbol, 'strike': strike,
#             'option_type': option_type, 'event': event, 'message': message,
#             'price': price, 'quantity': quantity})

#     def _calculate_statistics(self, results: Dict) -> Dict:
#         trades_df = pd.DataFrame(results['trades'])
#         if trades_df.empty:
#             return {'error': 'No trades executed'}

#         completed = trades_df[trades_df['exit_time'].notna()].copy()
#         if completed.empty:
#             return {'error': 'No completed trades'}

#         total_pnl = completed['pnl'].sum()
#         winning = completed[completed['pnl'] > 0]
#         losing = completed[completed['pnl'] < 0]

#         stats = {
#             'total_trades': len(completed),
#             'total_pnl': total_pnl,
#             'winning_trades': len(winning),
#             'losing_trades': len(losing),
#             'win_rate': len(winning) / len(completed) * 100 if len(completed) > 0 else 0,
#             'avg_pnl_per_trade': completed['pnl'].mean(),
#             'max_profit': completed['pnl'].max(),
#             'max_loss': completed['pnl'].min(),
#             'avg_winner': winning['pnl'].mean() if len(winning) > 0 else 0,
#             'avg_loser': losing['pnl'].mean() if len(losing) > 0 else 0,
#             'profit_factor': abs(winning['pnl'].sum() / losing['pnl'].sum())
#             if losing['pnl'].sum() != 0 else float('inf')}

#         for reason in ['SL_HIT', 'EOD_SQUAREOFF', 'RMS_POSITIVE_QTY', 'RMS_INVALID_STRIKE']:
#             count = len(completed[completed['exit_reason'] == reason])
#             stats[f'{reason.lower()}_count'] = count
#             stats[f'{reason.lower()}_rate'] = count / len(completed) * 100

#         daily_pnl = pd.Series(results['daily_pnl'])
#         if len(daily_pnl) > 0:
#             stats['trading_days'] = len(daily_pnl)
#             stats['profitable_days'] = len(daily_pnl[daily_pnl > 0])
#             stats['losing_days'] = len(daily_pnl[daily_pnl < 0])
#             stats['best_day'] = daily_pnl.max()
#             stats['worst_day'] = daily_pnl.min()
#             stats['avg_daily_pnl'] = daily_pnl.mean()
#             if daily_pnl.std() != 0:
#                 stats['sharpe_ratio'] = (daily_pnl.mean() / daily_pnl.std()) * np.sqrt(252)
#             else:
#                 stats['sharpe_ratio'] = 0

#         return stats


# # =============================================================================
# # STEP 5: LOAD YOUR CSV DATA
# # =============================================================================

# print("STEP 1: Loading CSV files...")

# # ── Spot data ──────────────────────────────────────────────────────────────
# spot_df = pd.read_csv('nifty_spot_15_04_2026.csv', parse_dates=['date_time'])

# spot_df = spot_df.rename(columns={
#     'date_time': 'date',
#     'index': 'symbol'
# })
# spot_df.set_index('date', inplace=True)

# # Normalise index to tz-naive datetime
# spot_df.index = pd.to_datetime(
#     spot_df.index.map(lambda x: x[1] if isinstance(x, tuple) else x)
# ).tz_localize(None)   # FIX: strip tz here

# spot_df = spot_df[['symbol', 'open', 'high', 'low', 'close', 'volume']].copy()
# spot_df['volume'] = 100_000   # dummy volume for VWAP

# print(f"Spot data loaded: {spot_df.shape}")

# # ── Options data ───────────────────────────────────────────────────────────
# options_df = pd.read_csv(
#     'nifty_opt_15_04_2026.csv',
#     parse_dates=['date_time', 'expiry_date'])

# options_df = options_df.rename(columns={
#     'date_time': 'date',
#     'expiry_date': 'expiry',
#     'index': 'symbol'
# })
# options_df.set_index('date', inplace=True)

# # Normalise index to tz-naive datetime
# options_df.index = pd.to_datetime(
#     options_df.index.map(lambda x: x[1] if isinstance(x, tuple) else x)
# ).tz_localize(None)   # FIX: strip tz here

# # FIX: Normalise expiry column to tz-naive
# options_df['expiry'] = pd.to_datetime(options_df['expiry']).dt.tz_localize(None)

# options_df = options_df[
#     ['symbol', 'strike', 'option_type', 'expiry',
#      'open', 'high', 'low', 'close', 'volume']
# ].copy()

# # Filter for the specific expiry
# current_expiry = pd.Timestamp('2026-04-13')   # tz-naive, matches stripped index
# options_df = options_df[options_df['expiry'] == current_expiry].copy()

# print(f"\nOptions data loaded: {options_df.shape}")


# # =============================================================================
# # STEP 6: RUN BACKTEST
# # =============================================================================


# print("STEP 2: Initializing Backtester...")

# params = StrategyParams(
#     symbols=['NIFTY'],
#     capital_per_symbol=10_000_000,
#     threshold_percentage=10.0,
#     sl_percentage=30.0,
#     num_strikes=10,
#     max_open_strikes_per_leg=3,
#     lot_sizes={'NIFTY': 65},
#     strike_diffs={'NIFTY': 50.0},
# )

# loader = DataFrameLoader(spot_df, options_df)
# backtester = OSTRADBacktester(params, loader)


# # FIX: Use tz-naive datetimes — timezone stripping is handled internally
# start_date = datetime(2026, 4, 15)
# end_date = datetime(2026, 4, 15, 23, 59, 59)

# results = backtester.run_backtest(start_date, end_date)


# # =============================================================================
# # STEP 7: PRINT RESULTS
# # =============================================================================


# print("BACKTEST RESULTS")
# print("=" * 60)

# stats = results['statistics']

# if 'error' in stats:
#     print(f"Error: {stats['error']}")
# else:
#     print(f"\nTotal Trades:      {stats.get('total_trades', 0)}")
#     print(f"Total P&L:         ₹{stats.get('total_pnl', 0):,.2f}")
#     print(f"Win Rate:          {stats.get('win_rate', 0):.1f}%")
#     print(f"Profit Factor:     {stats.get('profit_factor', 0):.2f}")
#     print(f"\nMax Profit:        ₹{stats.get('max_profit', 0):,.2f}")
#     print(f"Max Loss:          ₹{stats.get('max_loss', 0):,.2f}")
#     print(f"Avg Winner:        ₹{stats.get('avg_winner', 0):,.2f}")
#     print(f"Avg Loser:         ₹{stats.get('avg_loser', 0):,.2f}")
#     print(f"\nSL Hit Rate:       {stats.get('sl_hit_rate', 0):.1f}%")
#     print(f"EOD Squareoff Rate:{stats.get('eod_squareoff_rate', 0):.1f}%")
#     print(f"\nSharpe Ratio:      {stats.get('sharpe_ratio', 0):.2f}")

# print("\n" + "=" * 60)
# print("Trade Log (first 100 entries):")
# print("=" * 60)
# for entry in backtester.trade_log[:200]:
#     print(f"{entry['timestamp'].strftime('%Y-%m-%d %H:%M')} | "
#           f"{entry['symbol']} {entry['strike']}{entry['option_type']} | "
#           f"{entry['event']}: {entry['message']}")

STEP 1: Loading CSV files...
Spot data loaded: (375, 6)

Options data loaded: (0, 9)
STEP 2: Initializing Backtester...
OSTRAD BACKTEST

Processing NIFTY...
  Loaded 375 spot candles
  Loaded 0 option records
  Prepared 0 straddle combinations
Running backtest for NIFTY...
  2026-04-15: P&L = ₹0.00
BACKTEST RESULTS
Error: No trades executed

Trade Log (first 100 entries):


In [2]:
"""
OSTRAD Backtest Runner
"""

import pandas as pd
from datetime import datetime
from ostrad_backtest import StrategyParams, DataFrameLoader, OSTRADBacktester


def run_backtest(
    spot_file: str,
    options_file: str,
    start_date: datetime,
    end_date: datetime,
    symbol: str = 'SENSEX',
    quantity: int = 800
):
    
    print("=" * 60)
    print("LOADING DATA...")
    print("=" * 60)

    # Load spot data
    spot_df = pd.read_csv(spot_file, parse_dates=['date_time'])
    spot_df.set_index('date_time', inplace=True)
    spot_df = spot_df.rename(columns={'index': 'symbol'})
    
    # Strip timezone if present
    if spot_df.index.tz is not None:
        spot_df.index = spot_df.index.tz_localize(None)
    
    spot_df = spot_df[['symbol', 'open', 'high', 'low', 'close', 'volume']].copy()
    spot_df['volume'] = spot_df['volume'].fillna(100000)
    
    print(f"Spot data: {spot_df.shape[0]} rows")
    print(f"Date range: {spot_df.index.min()} to {spot_df.index.max()}")

    # Load options data
    options_df = pd.read_csv(options_file, parse_dates=['date_time', 'expiry_date'])
    options_df.set_index('date_time', inplace=True)
    options_df = options_df.rename(columns={'index': 'symbol', 'expiry_date': 'expiry'})
    
    # Strip timezone if present
    if options_df.index.tz is not None:
        options_df.index = options_df.index.tz_localize(None)
    
    if options_df['expiry'].dt.tz is not None:
        options_df['expiry'] = options_df['expiry'].dt.tz_localize(None)
    
    options_df = options_df[['symbol', 'strike', 'option_type', 'expiry', 
                              'open', 'high', 'low', 'close', 'volume']].copy()
    
    print(f"Options data: {options_df.shape[0]} rows")
    print(f"Expiries: {options_df['expiry'].unique()}")

    # Configure params
    params = StrategyParams(
        symbols=[symbol],
        lot_sizes={symbol: 20},
        strike_diffs={symbol: 50.0},
        quantity_per_strike={symbol: quantity},
        capital_per_symbol=10000000,
        threshold_percentage=10.0,
        sl_percentage=30.0,
        sl_diff_percentage=2.0,
        num_strikes=3,
        max_open_strikes_per_leg=3,
        min_premium_percent=0.055,
        new_position_margin_limit=75.0,
        hedge_margin_limit=60.0,
        hedge_strike_diff_percent=2.5,
        hedge_strike_diff_percent_2=5.0,
        time_frame=5,
        start_time="09:25:00",
        entry_end_time="14:30:00",
        sl_modification_time="10:00:00",
        square_off_time="15:20:00",
        active_days_to_expiry=[0, 1, 2],
    )

    print(f"\nSymbol: {symbol}")
    print(f"Quantity per strike: {quantity}")

    # Run backtest
    loader = DataFrameLoader(spot_df, options_df)
    backtester = OSTRADBacktester(params, loader)
    
    print(f"\nBacktest: {start_date.date()} to {end_date.date()}")
    
    results = backtester.run_backtest(start_date, end_date)
    backtester.print_results(results)
    backtester.print_trade_log(limit=30)

    # Save results
    trades_df = pd.DataFrame(results['trades'])
    if not trades_df.empty:
        trades_df.to_csv('backtest_trades.csv', index=False)
        print(f"\nTrades saved to backtest_trades.csv")

    daily_pnl_df = pd.DataFrame(list(results['daily_pnl'].items()), columns=['date', 'pnl'])
    if not daily_pnl_df.empty:
        daily_pnl_df.to_csv('backtest_daily_pnl.csv', index=False)
        print(f"Daily P&L saved to backtest_daily_pnl.csv")
    
    return results


if __name__ == "__main__":
    # =====================================================
    # CONFIGURE YOUR BACKTEST HERE
    # =====================================================
    
    # CSV file paths
    SPOT_FILE = 'sensex_spot_15_04_2026.csv'
    OPTIONS_FILE = 'sensex_opt_15_04_2026.csv'
    
    # Date range (change these for your data)
    START_DATE = datetime(2026, 4, 15)
    END_DATE = datetime(2026, 4, 15)
    
    # Symbol and quantity
    SYMBOL = 'SENSEX'
    QUANTITY = 800
    
    # =====================================================
    # RUN BACKTEST
    # =====================================================
    
    results = run_backtest(
        spot_file=SPOT_FILE,
        options_file=OPTIONS_FILE,
        start_date=START_DATE,
        end_date=END_DATE,
        symbol=SYMBOL,
        quantity=QUANTITY
    )

LOADING DATA...
Spot data: 375 rows
Date range: 2026-04-15 09:15:00 to 2026-04-15 15:29:00
Options data: 197797 rows
Expiries: <DatetimeArray>
['2026-04-16 00:00:00', '2026-04-23 00:00:00', '2026-04-30 00:00:00',
 '2026-05-07 00:00:00', '2026-05-14 00:00:00', '2026-05-27 00:00:00',
 '2026-06-25 00:00:00']
Length: 7, dtype: datetime64[ns]

Symbol: SENSEX
Quantity per strike: 800

Backtest: 2026-04-15 to 2026-04-15
OSTRAD BACKTEST
Date Range: 2026-04-15 to 2026-04-15
Symbols: ['SENSEX']
Quantity per strike: {'SENSEX': 800}

Processing SENSEX...
  Loaded 375 spot candles
  Loaded 197797 option records
  Available expiries: ['2026-04-16', '2026-04-23', '2026-04-30', '2026-05-07', '2026-05-14', '2026-05-27', '2026-06-25']
Running backtest for SENSEX...
  2026-04-15: P&L = ₹29,680.00

BACKTEST RESULTS

SUMMARY
Total Trades:        4
Total P&L:           ₹29,680.00
Win Rate:            25.0%
Profit Factor:       1.71

TRADE BREAKDOWN
Winning Trades:      1
Losing Trades:       3
Max Profit:  